# GeneTropica Phase 14 — Molecular Dynamics Simulation

**Three-drug mechanism comparison on DENV NS5 RdRp (PDB 5CCV, Chain A)**

| Drug | Consensus Rank | Mechanism | Purpose |
|------|---------------|-----------|----------|
| Celecoxib | #1 (0.6846) | COX-2 inhibitor | Pipeline's top prediction |
| Methotrexate | #3 (0.4930) | DHFR / host-directed | Tests indirect mechanism |
| Dasabuvir | #17 (0.3758) | Non-nucleoside RdRp | Direct polymerase binder |

**Protocol:** 50 ns all-atom MD per drug using GROMACS + ACPYPE (GAFF2)

**Runtime estimate:** 3–9 days total (T4 GPU)

---
Russell Young — British School Jakarta

In [ ]:
# ============================================================
# Cell 1: Install GROMACS and dependencies (~3 min)
# ============================================================
import subprocess, sys, os

print('Installing GROMACS...')
!apt-get update -qq
!apt-get install -y -qq gromacs > /dev/null 2>&1

print('Installing ACPYPE for ligand parametrisation...')
!pip install -q acpype

print('Installing PDBFixer for structure repair...')
!pip install -q pdbfixer

print('Installing analysis tools...')
!pip install -q MDAnalysis matplotlib numpy

# Verify installation
!gmx --version | head -5
print()
try:
    import pdbfixer
    print(f'PDBFixer {pdbfixer.__version__} installed')
except:
    print('PDBFixer installed (version check skipped)')
print('\n=== Installation complete ===')

In [ ]:
# ============================================================
# Cell 2: Upload input files (select ALL 8 files)
# ============================================================
from google.colab import files
import os

WORKDIR = '/content/md_simulation'
os.makedirs(WORKDIR, exist_ok=True)
os.chdir(WORKDIR)

print('Please upload ALL 8 files from data/md_simulation/:')
print('  From celecoxib/input/  : protein_5CCV.pdb, celecoxib_docked.mol2')
print('  From methotrexate/input/: methotrexate_docked.mol2')
print('  From dasabuvir/input/  : dasabuvir_docked.mol2')
print('  From mdp/              : em.mdp, nvt.mdp, npt.mdp, md.mdp')
print()

uploaded = files.upload()

REQUIRED = [
    'protein_5CCV.pdb',
    'celecoxib_docked.mol2', 'methotrexate_docked.mol2', 'dasabuvir_docked.mol2',
    'em.mdp', 'nvt.mdp', 'npt.mdp', 'md.mdp',
]
missing = [f for f in REQUIRED if not os.path.exists(f)]
if missing:
    print(f'\n*** MISSING FILES: {missing} ***')
    print('Please re-upload the missing files.')
else:
    print(f'\n=== All {len(REQUIRED)} files uploaded successfully ===')
    for f in sorted(os.listdir('.')):
        if not f.startswith('.'):
            print(f'  {f} ({os.path.getsize(f):,} bytes)')

In [ ]:
# ============================================================
# Cell 3: Verify GPU is available
# ============================================================
import subprocess

result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if result.returncode == 0:
    for line in result.stdout.split('\n')[:10]:
        print(line)
    print('\n=== GPU detected — MD will use GPU acceleration ===')
else:
    print('*** WARNING: No GPU detected! ***')
    print('Go to Runtime > Change runtime type > T4 GPU > Save')
    print('Then re-run from Cell 1.')

In [ ]:
# ============================================================
# Cell 4: Helper functions (shared by all 3 drugs)
# ============================================================
import matplotlib.pyplot as plt
import numpy as np
import os, subprocess, shutil, glob, re

WORKDIR = '/content/md_simulation'


def run_cmd(cmd, label='', check=True):
    """Run a shell command, print output on failure."""
    if label:
        print(f'  [{label}]')
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if check and result.returncode != 0:
        print(f'  STDOUT: {result.stdout[-500:]}')
        print(f'  STDERR: {result.stderr[-2000:]}')
        raise RuntimeError(f'Command failed: {cmd}')
    return result


def plot_xvg(xvg_path, title, xlabel, ylabel, save_path):
    """Parse GROMACS .xvg file and plot."""
    x, y = [], []
    with open(xvg_path) as f:
        for line in f:
            if line.startswith(('#', '@')):
                continue
            parts = line.split()
            if len(parts) >= 2:
                x.append(float(parts[0]))
                y.append(float(parts[1]))
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(x, y, linewidth=0.8)
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.grid(True, alpha=0.3)
    fig.tight_layout()
    fig.savefig(save_path, dpi=150)
    plt.show()
    print(f'  Saved: {save_path}')


def fix_pdb_for_gromacs(pdb_in, pdb_out):
    """Prepare PDB for GROMACS pdb2gmx.
    
    Handles three issues with 5CCV Chain A:
    1. Use PDBFixer to add missing heavy atoms (e.g. ARG 839 missing
       CG — incomplete side chains from crystal structure).
    2. Rename HIS -> HIE (epsilon-protonated) to avoid interactive
       protonation prompts (24 HIS residues).
    3. Detect chain breaks (gaps in residue numbering from missing
       loops) and insert TER records so GROMACS knows where they are.
       5CCV Chain A has 3 breaks: 404->420, 453->478, 794->801
       (45 missing residues total).
    
    Used with pdb2gmx flags: -merge all -ter
    """
    from pdbfixer import PDBFixer
    from openmm.app import PDBFile
    
    # ---- Step 1: Add missing heavy atoms with PDBFixer ----
    print('  Running PDBFixer to repair incomplete residues...')
    fixer = PDBFixer(filename=pdb_in)
    
    # Must call findMissingResidues first (findMissingAtoms needs
    # the missingResidues attribute internally), then clear it so
    # PDBFixer doesn't try to fill the 45-residue loop gaps.
    fixer.findMissingResidues()
    fixer.missingResidues = {}  # Don't add missing loop residues
    fixer.findMissingAtoms()
    
    if fixer.missingAtoms:
        n_missing = sum(len(atoms) for atoms in fixer.missingAtoms.values())
        print(f'  PDBFixer: adding {n_missing} missing heavy atoms '
              f'in {len(fixer.missingAtoms)} residues:')
        for residue, atoms in fixer.missingAtoms.items():
            print(f'    {residue.name} {residue.id}: '
                  f'{[a.name for a in atoms]}')
    else:
        print('  PDBFixer: no missing atoms found.')
    
    # Don't add terminal-specific atoms (OXT etc.) —
    # GROMACS pdb2gmx with -ter handles termini itself.
    fixer.missingTerminals = {}
    fixer.addMissingAtoms()
    
    # Write intermediate PDB
    tmp_pdb = pdb_out + '.tmp'
    with open(tmp_pdb, 'w') as f:
        PDBFile.writeFile(fixer.topology, fixer.positions, f)
    
    # ---- Step 2: HIS->HIE rename + chain break TER insertion ----
    with open(tmp_pdb) as f:
        lines = f.readlines()
    
    # Collect ATOM/HETATM lines, rename HIS -> HIE
    atom_lines = []
    for line in lines:
        # Convert any HETATM to ATOM for GROMACS compatibility
        if line.startswith('HETATM'):
            line = 'ATOM  ' + line[6:]
        if line.startswith('ATOM'):
            if line[17:20] == 'HIS':
                line = line[:17] + 'HIE' + line[20:]
            atom_lines.append(line)
    
    # Detect chain breaks and insert TER records
    out = []
    prev_resnum = None
    breaks = []
    for line in atom_lines:
        resnum = int(line[22:26].strip())
        if resnum != prev_resnum:
            # New residue — check for gap
            if prev_resnum is not None and resnum > prev_resnum + 1:
                out.append('TER\n')
                breaks.append((prev_resnum, resnum, resnum - prev_resnum - 1))
            prev_resnum = resnum
        out.append(line)
    
    out.append('TER\n')
    out.append('END\n')
    
    with open(pdb_out, 'w') as f:
        f.writelines(out)
    
    # Clean up temp file
    os.remove(tmp_pdb)
    
    n_his = sum(1 for l in out if l.startswith('ATOM') and l[17:20] == 'HIE')
    n_atoms = sum(1 for l in out if l.startswith('ATOM'))
    print(f'  PDB cleaned: {n_atoms} atoms, {n_his} HIS->HIE renamed')
    if breaks:
        print(f'  Chain breaks detected ({len(breaks)}):')
        for prev_res, next_res, gap in breaks:
            print(f'    Residue {prev_res} -> {next_res} ({gap} missing)')
        print(f'  TER records inserted at each break for -merge all')
    else:
        print('  No chain breaks detected.')
    return pdb_out


def prepare_system(drug_name):
    """Prepare GROMACS system for one drug.
    
    Steps: fix PDB -> pdb2gmx -> ACPYPE ligand -> combine -> solvate -> ions -> index
    """
    drug_dir = os.path.join(WORKDIR, drug_name)
    os.makedirs(drug_dir, exist_ok=True)
    os.chdir(drug_dir)
    
    mol2_file = os.path.join(WORKDIR, f'{drug_name}_docked.mol2')
    protein_pdb = os.path.join(WORKDIR, 'protein_5CCV.pdb')
    
    print(f'\n{"="*60}')
    print(f'  PREPARING: {drug_name.upper()}')
    print(f'{"="*60}')
    
    # --- Step 1: Fix PDB for GROMACS ---
    print('\n[1/7] Cleaning protein PDB for GROMACS...')
    fixed_pdb = fix_pdb_for_gromacs(protein_pdb, 'protein_fixed.pdb')
    
    # --- Step 2: Process protein with pdb2gmx (AMBER99SB-ILDN) ---
    print('\n[2/7] Processing protein with pdb2gmx...')
    run_cmd(
        f'gmx pdb2gmx -f {fixed_pdb} -o protein.gro '
        f'-p topol.top -ignh -ff amber99sb-ildn -water tip3p '
        f'-merge all -ter',
        'pdb2gmx'
    )
    
    # --- Step 3: Parametrise ligand with ACPYPE ---
    print('\n[3/7] Parametrising ligand with ACPYPE (GAFF2)...')
    # Use Gasteiger charges (-c gas) — AM1-BCC (-c bcc) requires
    # AmberTools/sqm which is not available on Colab by default.
    # Gasteiger charges are adequate for comparative MD studies.
    # All 3 drugs use the same method so the comparison is fair.
    acpype_result = run_cmd(
        f'acpype -i {mol2_file} -c gas -n 0 -a gaff2',
        'acpype',
        check=False
    )
    if acpype_result.returncode != 0:
        print('  ACPYPE with gaff2 failed, trying gaff...')
        # Clean up failed attempt
        for d in glob.glob('*.acpype'):
            shutil.rmtree(d, ignore_errors=True)
        run_cmd(
            f'acpype -i {mol2_file} -c gas -n 0 -a gaff',
            'acpype-fallback'
        )
    
    # Find the ACPYPE output directory
    acpype_dirs = sorted(glob.glob('*.acpype'))
    if not acpype_dirs:
        raise FileNotFoundError('ACPYPE output directory not found!')
    acpype_dir = acpype_dirs[0]
    print(f'  ACPYPE output: {acpype_dir}')
    
    # Copy GROMACS topology files from ACPYPE
    lig_itp = glob.glob(os.path.join(acpype_dir, '*_GMX.itp'))
    lig_gro = glob.glob(os.path.join(acpype_dir, '*_GMX.gro'))
    if not lig_itp or not lig_gro:
        # Fallback: try any .itp/.gro (filter posre)
        lig_itp = [f for f in glob.glob(os.path.join(acpype_dir, '*.itp'))
                   if 'posre' not in f.lower()]
        lig_gro = glob.glob(os.path.join(acpype_dir, '*.gro'))
    
    shutil.copy(lig_itp[0], 'ligand.itp')
    shutil.copy(lig_gro[0], 'ligand.gro')
    print(f'  Ligand ITP: {os.path.basename(lig_itp[0])}')
    print(f'  Ligand GRO: {os.path.basename(lig_gro[0])}')
    
    # Copy atomtypes if present (GAFF2 custom types)
    at_files = glob.glob(os.path.join(acpype_dir, '*atomtypes*itp'))
    has_atomtypes = False
    if at_files:
        shutil.copy(at_files[0], 'ligand_atomtypes.itp')
        has_atomtypes = True
    
    # --- Step 4: Combine protein + ligand into complex.gro ---
    print('\n[4/7] Combining protein + ligand...')
    with open('protein.gro') as f:
        prot_lines = f.readlines()
    with open('ligand.gro') as f:
        lig_lines = f.readlines()
    
    prot_natoms = int(prot_lines[1].strip())
    lig_natoms = int(lig_lines[1].strip())
    total_atoms = prot_natoms + lig_natoms
    
    with open('complex.gro', 'w') as f:
        f.write(f'Protein-ligand complex: {drug_name}\n')
        f.write(f'{total_atoms}\n')
        for line in prot_lines[2:2+prot_natoms]:
            f.write(line)
        for line in lig_lines[2:2+lig_natoms]:
            f.write(line)
        f.write(prot_lines[-1])  # box vector
    print(f'  Combined: {prot_natoms} protein + {lig_natoms} ligand = {total_atoms} atoms')
    
    # --- Step 5: Update topology to include ligand ---
    print('\n[5/7] Updating topology...')
    with open('topol.top') as f:
        top_content = f.read()
    
    # Get ligand residue name from .itp
    lig_resname = 'LIG'
    in_atoms = False
    with open('ligand.itp') as f:
        for line in f:
            if '[ atoms ]' in line:
                in_atoms = True
                continue
            if in_atoms and line.strip() and not line.startswith(';'):
                parts = line.split()
                if len(parts) >= 4:
                    lig_resname = parts[3]
                break
    
    # Insert includes after forcefield.itp
    lines = top_content.split('\n')
    new_lines = []
    ff_done = False
    for line in lines:
        new_lines.append(line)
        if 'forcefield.itp' in line and not ff_done:
            if has_atomtypes:
                new_lines.append('#include "ligand_atomtypes.itp"')
            new_lines.append('#include "ligand.itp"')
            ff_done = True
    
    # Add ligand to [ molecules ]
    new_lines.append(f'{lig_resname}     1')
    
    with open('topol.top', 'w') as f:
        f.write('\n'.join(new_lines))
    print(f'  Ligand residue: {lig_resname}')
    
    # --- Step 6: Solvate ---
    print('\n[6/7] Box + solvate...')
    run_cmd(
        'gmx editconf -f complex.gro -o box.gro -c -d 1.2 -bt dodecahedron',
        'editconf'
    )
    run_cmd(
        'gmx solvate -cp box.gro -cs spc216.gro -o solvated.gro -p topol.top',
        'solvate'
    )
    
    # --- Step 7: Add ions ---
    print('\n[7/7] Adding ions to neutralise...')
    run_cmd(
        f'gmx grompp -f {os.path.join(WORKDIR, "em.mdp")} '
        f'-c solvated.gro -p topol.top -o ions.tpr -maxwarn 5',
        'grompp-ions'
    )
    run_cmd(
        'echo SOL | gmx genion -s ions.tpr -o system.gro '
        '-p topol.top -pname NA -nname CL -neutral',
        'genion'
    )
    
    # --- Create index groups ---
    # MDP files expect tc-grps = Protein_LIG Water_and_ions
    # make_ndx auto-names the merged group "Protein_Other", so we
    # post-process the ndx file to rename it to "Protein_LIG".
    print('\n  Creating index groups...')

    # List available groups
    result = run_cmd('echo q | gmx make_ndx -f system.gro', check=False)
    ndx_out = result.stdout + result.stderr

    # Print all detected groups for debugging
    print('  Default groups:')
    for line in ndx_out.split('\n'):
        line = line.strip()
        if re.match(r'\d+\s+\S+', line) and ':' in line:
            print(f'    {line}')

    # Find the "Other" group (contains ligand)
    other_grp = None
    other_name = None
    for line in ndx_out.split('\n'):
        line = line.strip()
        m = re.match(r'(\d+)\s+(Other|LIG|UNK|MOL)\b', line, re.IGNORECASE)
        if m:
            other_grp = m.group(1)
            other_name = m.group(2)
            print(f'  Ligand index group: {line}')
            break

    if other_grp is None:
        other_grp = '13'
        other_name = 'Other'
        print(f'  Using default ligand group: {other_grp}')

    # Verify Water_and_ions exists in default groups
    has_water_ions = ('Water_and_ions' in ndx_out or
                      'Water_and_Ions' in ndx_out)
    if has_water_ions:
        print('  Water_and_ions group: found')
    else:
        print('  WARNING: Water_and_ions not found in defaults')

    # Create Protein_LIG group: Protein | Ligand
    ndx_cmds = f'1 | {other_grp}\\nq\\n'
    run_cmd(
        f'printf "{ndx_cmds}" | gmx make_ndx -f system.gro -o index.ndx',
        'make_ndx',
        check=False
    )

    # Post-process index.ndx: rename the auto-generated combined
    # group (e.g. "Protein_Other") to "Protein_LIG" so it matches
    # the tc-grps in nvt.mdp, npt.mdp, and md.mdp.
    if os.path.exists('index.ndx'):
        with open('index.ndx') as f:
            ndx = f.read()

        # Rename the combined group to Protein_LIG
        renamed = False
        for auto_name in [f'Protein_{other_name}',
                          'Protein_Other', 'Protein_UNK',
                          'Protein_MOL']:
            if f'[ {auto_name} ]' in ndx:
                ndx = ndx.replace(f'[ {auto_name} ]',
                                  '[ Protein_LIG ]')
                print(f'  Renamed group: {auto_name} -> Protein_LIG')
                renamed = True
                break

        if not renamed:
            if '[ Protein_LIG ]' in ndx:
                print('  Protein_LIG group already exists')
            else:
                print('  WARNING: Could not find combined group to '
                      'rename. NVT/NPT may fail.')

        # Handle Water_and_ions capitalisation variants
        if ('[ Water_and_Ions ]' in ndx and
                '[ Water_and_ions ]' not in ndx):
            ndx = ndx.replace('[ Water_and_Ions ]',
                              '[ Water_and_ions ]')
            print('  Fixed capitalisation: Water_and_Ions -> '
                  'Water_and_ions')

        with open('index.ndx', 'w') as f:
            f.write(ndx)

        # Final verification
        groups_found = re.findall(r'\[ (\S+) \]', ndx)
        print(f'  Index groups ({len(groups_found)}): '
              f'{", ".join(groups_found[-5:])} ...')
        for needed in ['Protein_LIG', 'Water_and_ions']:
            status = 'found' if needed in groups_found else 'MISSING'
            print(f'  {needed}: {status}')

    print(f'\n=== {drug_name.upper()} system prepared ===')
    return drug_dir


def run_equilibration(drug_name):
    """Run EM + NVT + NPT equilibration."""
    drug_dir = os.path.join(WORKDIR, drug_name)
    os.chdir(drug_dir)
    
    print(f'\n{"="*60}')
    print(f'  EQUILIBRATING: {drug_name.upper()}')
    print(f'{"="*60}')
    
    # --- Energy Minimisation ---
    print('\n[EM] Energy minimisation...')
    run_cmd(
        f'gmx grompp -f {os.path.join(WORKDIR, "em.mdp")} '
        f'-c system.gro -p topol.top -o em.tpr -maxwarn 5',
        'grompp-em'
    )
    run_cmd('gmx mdrun -v -deffnm em -nb gpu', 'mdrun-em')
    
    # Plot EM potential energy
    run_cmd('echo Potential | gmx energy -f em.edr -o em_potential.xvg',
            'energy-em', check=False)
    if os.path.exists('em_potential.xvg'):
        plot_xvg('em_potential.xvg',
                 f'{drug_name} — Energy Minimisation',
                 'Step', 'Potential Energy (kJ/mol)',
                 f'em_energy_{drug_name}.png')
    
    # --- NVT Equilibration ---
    print('\n[NVT] NVT equilibration (100 ps, 300K)...')
    run_cmd(
        f'gmx grompp -f {os.path.join(WORKDIR, "nvt.mdp")} '
        f'-c em.gro -r em.gro -p topol.top -o nvt.tpr -n index.ndx -maxwarn 5',
        'grompp-nvt'
    )
    run_cmd('gmx mdrun -deffnm nvt -nb gpu', 'mdrun-nvt')
    
    # Plot NVT temperature
    run_cmd('echo Temperature | gmx energy -f nvt.edr -o nvt_temp.xvg',
            'energy-nvt', check=False)
    if os.path.exists('nvt_temp.xvg'):
        plot_xvg('nvt_temp.xvg',
                 f'{drug_name} — NVT Temperature',
                 'Time (ps)', 'Temperature (K)',
                 f'nvt_temp_{drug_name}.png')
    
    # --- NPT Equilibration ---
    print('\n[NPT] NPT equilibration (100 ps, 300K, 1 bar)...')
    run_cmd(
        f'gmx grompp -f {os.path.join(WORKDIR, "npt.mdp")} '
        f'-c nvt.gro -r nvt.gro -t nvt.cpt -p topol.top '
        f'-o npt.tpr -n index.ndx -maxwarn 5',
        'grompp-npt'
    )
    run_cmd('gmx mdrun -deffnm npt -nb gpu', 'mdrun-npt')
    
    # Plot NPT pressure / density
    run_cmd('echo Density | gmx energy -f npt.edr -o npt_density.xvg',
            'energy-npt-density', check=False)
    if os.path.exists('npt_density.xvg'):
        plot_xvg('npt_density.xvg',
                 f'{drug_name} — NPT Density',
                 'Time (ps)', u'Density (kg/m\u00b3)',
                 f'npt_density_{drug_name}.png')
    
    print(f'\n=== {drug_name.upper()} equilibration complete ===')


def run_production_full(drug_name):
    """Full 50 ns production in one shot."""
    drug_dir = os.path.join(WORKDIR, drug_name)
    os.chdir(drug_dir)
    
    print(f'\n{"="*60}')
    print(f'  PRODUCTION (50 ns): {drug_name.upper()}')
    print(f'{"="*60}')
    
    run_cmd(
        f'gmx grompp -f {os.path.join(WORKDIR, "md.mdp")} '
        f'-c npt.gro -t npt.cpt -p topol.top -o md.tpr -n index.ndx -maxwarn 5',
        'grompp-md'
    )
    print('  Starting 50 ns production run...')
    print('  Estimated time: 24-72 hours on T4 GPU.')
    run_cmd('gmx mdrun -deffnm md -nb gpu -v', 'mdrun-production')
    print(f'\n=== {drug_name.upper()} production complete ===')


def run_production_chunked(drug_name, chunk_ns=10, total_ns=50):
    """Production in chunks for Colab Free tier."""
    drug_dir = os.path.join(WORKDIR, drug_name)
    os.chdir(drug_dir)
    
    n_chunks = total_ns // chunk_ns
    steps_per_chunk = int(chunk_ns * 1e6 / 2)  # dt=0.002 ps
    
    print(f'\n{"="*60}')
    print(f'  PRODUCTION (CHUNKED {n_chunks}x{chunk_ns}ns): {drug_name.upper()}')
    print(f'{"="*60}')
    
    for chunk in range(n_chunks):
        print(f'\n--- Chunk {chunk+1}/{n_chunks} '
              f'({chunk*chunk_ns}-{(chunk+1)*chunk_ns} ns) ---')
        
        if chunk == 0:
            # First chunk: create modified MDP for chunk length
            with open(os.path.join(WORKDIR, 'md.mdp')) as f:
                mdp = f.read()
            mdp = mdp.replace('nsteps              = 25000000',
                              f'nsteps              = {steps_per_chunk}')
            with open('md_chunk.mdp', 'w') as f:
                f.write(mdp)
            run_cmd(
                f'gmx grompp -f md_chunk.mdp '
                f'-c npt.gro -t npt.cpt -p topol.top '
                f'-o md.tpr -n index.ndx -maxwarn 5',
                f'grompp-chunk{chunk+1}'
            )
        else:
            # Extend from checkpoint
            run_cmd(
                f'gmx convert-tpr -s md.tpr '
                f'-extend {chunk_ns * 1000} -o md.tpr',
                f'extend-chunk{chunk+1}'
            )
        
        print(f'  Running chunk {chunk+1}... (~5-14 hours on T4)')
        run_cmd(
            'gmx mdrun -deffnm md -nb gpu -v -cpi md.cpt',
            f'mdrun-chunk{chunk+1}'
        )
        print(f'  Chunk {chunk+1} complete!')
        
        # Save checkpoint to Drive
        try:
            dd = f'/content/drive/MyDrive/GeneTropica_MD/{drug_name}'
            os.makedirs(dd, exist_ok=True)
            for fn in ['md.cpt', 'md.xtc', 'md.edr', 'md.log']:
                if os.path.exists(fn):
                    shutil.copy(fn, dd)
            print(f'  Checkpoint saved to Drive: {dd}')
        except Exception as e:
            print(f'  (Drive save skipped: {e})')
    
    print(f'\n=== {drug_name.upper()} chunked production complete ===')


def package_results(drug_name):
    """Package results into tar.gz for download."""
    drug_dir = os.path.join(WORKDIR, drug_name)
    os.chdir(drug_dir)
    
    result_files = [
        'md.xtc', 'md.tpr', 'md.gro', 'md.edr', 'topol.top',
        'md.cpt', 'md.log', 'index.ndx',
        'em.gro', 'npt.gro', 'system.gro', 'ligand.itp',
    ]
    result_files += glob.glob('*.png')
    if os.path.exists('ligand_atomtypes.itp'):
        result_files.append('ligand_atomtypes.itp')
    
    existing = [f for f in result_files if os.path.exists(f)]
    tar_name = f'md_results_{drug_name}.tar.gz'
    tar_path = os.path.join(WORKDIR, tar_name)
    
    run_cmd(f'tar -czf {tar_path} {" ".join(existing)}',
            f'package-{drug_name}')
    
    size_mb = os.path.getsize(tar_path) / 1e6
    print(f'  Packaged: {tar_name} ({size_mb:.1f} MB, {len(existing)} files)')
    return tar_path


print('=== Helper functions loaded ===')

In [ ]:
# ============================================================
# Cell 5: (Optional) Mount Google Drive for backup
# ============================================================
# Uncomment to enable automatic checkpoint saves to Drive.

# from google.colab import drive
# drive.mount('/content/drive')
# print('Drive mounted. Checkpoints save to /content/drive/MyDrive/GeneTropica_MD/')

In [ ]:
# ============================================================
# Cell 6: Prepare all 3 drug systems (~15 min total)
# ============================================================
os.chdir(WORKDIR)

DRUGS = ['celecoxib', 'methotrexate', 'dasabuvir']

for drug in DRUGS:
    prepare_system(drug)

print('\n' + '='*60)
print('  ALL 3 SYSTEMS PREPARED SUCCESSFULLY')
print('='*60)

In [ ]:
# ============================================================
# Cell 7: Equilibrate all 3 systems (EM + NVT + NPT) (~90 min)
# ============================================================
# Verify plots after each drug:
#   EM energy  : should drop steeply then flatten
#   NVT temp   : should oscillate around 300 K
#   NPT density: should stabilise near 1000 kg/m3

for drug in DRUGS:
    run_equilibration(drug)

print('\n' + '='*60)
print('  ALL 3 SYSTEMS EQUILIBRATED')
print('  Check the plots above before proceeding!')
print('='*60)

In [ ]:
# ============================================================
# Cell 8: CHOOSE YOUR PRODUCTION MODE
# ============================================================
#
# OPTION A: 'full'    — 50 ns in one shot (Colab Pro)
# OPTION B: 'chunked' — 5 x 10 ns each (Colab Free, RECOMMENDED)

PRODUCTION_MODE = 'chunked'  # 'full' or 'chunked'

print(f'Production mode: {PRODUCTION_MODE}')
if PRODUCTION_MODE == 'chunked':
    print('Each drug: 5 chunks of 10 ns. Download between drugs.')
else:
    print('Full 50 ns per drug. Ensure stable connection.')

In [ ]:
# ============================================================
# Cell 9: CELECOXIB — Production MD
# ============================================================
# Consensus rank #1 (0.6846, ML 0.7026)
# Mechanism: COX-2 selective inhibitor
# Question: Does the pipeline's best prediction actually bind NS5 RdRp?

if PRODUCTION_MODE == 'full':
    run_production_full('celecoxib')
else:
    run_production_chunked('celecoxib', chunk_ns=10, total_ns=50)

In [ ]:
# ============================================================
# Cell 10: Package and download CELECOXIB results
# ============================================================
tar_path = package_results('celecoxib')

try:
    files.download(tar_path)
    print('Download started for md_results_celecoxib.tar.gz')
except Exception as e:
    print(f'Auto-download failed: {e}')
    print(f'Manual: Files sidebar > {tar_path}')

try:
    dd = '/content/drive/MyDrive/GeneTropica_MD/'
    os.makedirs(dd, exist_ok=True)
    shutil.copy(tar_path, dd)
    print(f'Saved to Drive: {dd}')
except:
    pass

print('\n>>> Download celecoxib results before running methotrexate! <<<')

In [ ]:
# ============================================================
# Cell 11: METHOTREXATE — Production MD
# ============================================================
# Consensus rank #3 (0.4930, ML 0.4834)
# Mechanism: DHFR / host-directed
# Question: Can a host-directed drug also bind the polymerase directly?

if PRODUCTION_MODE == 'full':
    run_production_full('methotrexate')
else:
    run_production_chunked('methotrexate', chunk_ns=10, total_ns=50)

In [ ]:
# ============================================================
# Cell 12: Package and download METHOTREXATE results
# ============================================================
tar_path = package_results('methotrexate')

try:
    files.download(tar_path)
    print('Download started for md_results_methotrexate.tar.gz')
except Exception as e:
    print(f'Auto-download failed: {e}')
    print(f'Manual: Files sidebar > {tar_path}')

try:
    dd = '/content/drive/MyDrive/GeneTropica_MD/'
    os.makedirs(dd, exist_ok=True)
    shutil.copy(tar_path, dd)
    print(f'Saved to Drive: {dd}')
except:
    pass

print('\n>>> Download methotrexate results before running dasabuvir! <<<')

In [ ]:
# ============================================================
# Cell 13: DASABUVIR — Production MD
# ============================================================
# Consensus rank #17 (0.3758)
# Mechanism: Non-nucleoside RdRp inhibitor (PMID 37632595)
# Question: Does cross-family RdRp conservation enable HCV->dengue binding?

if PRODUCTION_MODE == 'full':
    run_production_full('dasabuvir')
else:
    run_production_chunked('dasabuvir', chunk_ns=10, total_ns=50)

In [ ]:
# ============================================================
# Cell 14: Package and download DASABUVIR results
# ============================================================
tar_path = package_results('dasabuvir')

try:
    files.download(tar_path)
    print('Download started for md_results_dasabuvir.tar.gz')
except Exception as e:
    print(f'Auto-download failed: {e}')
    print(f'Manual: Files sidebar > {tar_path}')

try:
    dd = '/content/drive/MyDrive/GeneTropica_MD/'
    os.makedirs(dd, exist_ok=True)
    shutil.copy(tar_path, dd)
    print(f'Saved to Drive: {dd}')
except:
    pass

print('\n=== ALL THREE SIMULATIONS COMPLETE ===')
print('You should now have 3 tar.gz files. Extract and run Part 2.')

## Disconnection Recovery

If Colab disconnects during production:

1. Re-run **Cell 1** (reinstall GROMACS)
2. Re-run **Cell 2** (re-upload 8 files)
3. Re-run **Cell 3** (GPU check)
4. Re-run **Cell 4** (helper functions)
5. **Skip** Cells 6-7 if checkpoint exists
6. Jump to the drug cell where disconnection occurred
7. Chunked mode resumes from last checkpoint automatically

Use **Cell 16** below to check which checkpoints exist.

In [ ]:
# ============================================================
# Cell 16: Recovery — check checkpoint status
# ============================================================
for drug in ['celecoxib', 'methotrexate', 'dasabuvir']:
    dd = os.path.join(WORKDIR, drug)
    cpt = os.path.join(dd, 'md.cpt')
    npt = os.path.join(dd, 'npt.gro')
    if os.path.exists(cpt):
        print(f'{drug}: Production checkpoint found. Resume from production cell.')
    elif os.path.exists(npt):
        print(f'{drug}: Equilibration done. Start production cell.')
    else:
        print(f'{drug}: No checkpoint. Re-run Cell 6 (prepare) + Cell 7 (equilibrate).')